# Boids: flocking on a continuous space

A classic Boids flocking model. Each boid steers gently toward the average
position of the neighbours within its vision radius and keeps drifting. The
space is continuous and toroidal, so boids wrap around the edges.

In [ ]:
#!pip install mesa==2.3.4

In [ ]:
import mesa
import numpy as np
from mesa.time import RandomActivation
from mesa.space import ContinuousSpace

In [ ]:
class Boid(mesa.Agent):
    def __init__(self, unique_id, model, pos, velocity):
        super().__init__(unique_id, model)
        self.pos = np.array(pos, dtype=float)
        self.velocity = velocity

    def step(self):
        neighbors = self.model.space.get_neighbors(self.pos, 5, include_center=False)
        if neighbors:
            center = np.mean([n.pos for n in neighbors], axis=0)
            self.velocity = self.velocity + 0.05 * (center - self.pos)
        new_pos = self.pos + self.velocity
        self.model.space.move_agent(self, new_pos)

In [ ]:
class BoidFlockers(mesa.Model):
    def __init__(self, N=20, width=100, height=100, seed=None):
        super().__init__(seed=seed)
        self.num_agents = N
        self.space = ContinuousSpace(width, height, torus=True)
        self.schedule = RandomActivation(self)
        for _ in range(N):
            pos = np.array([self.random.random() * width,
                            self.random.random() * height])
            velocity = np.random.random(2) * 2 - 1
            boid = Boid(self.next_id(), self, pos, velocity)
            self.schedule.add(boid)
            self.space.place_agent(boid, pos)

    def step(self):
        self.schedule.step()

In [ ]:
model = BoidFlockers(20)
for _ in range(10):
    model.step()

print("steps:", model.schedule.steps, "| boids:", len(model.schedule.agents))